### 1. Imports e Configurações

In [1]:
import pandas as pd
import numpy as np
import socceraction.spadl as spd
import socceraction.spadl.config as spadlconfig
from socceraction.spadl.base import _fix_clearances, _add_dribbles

### 2. Parse de Eventos

In [2]:
def _parse_pass_event(row):
    a = "pass"
    r = str(row.get("result", "") or "").upper()
    b = str(row.get("body_part_type", "") or "").upper()
    if r == "COMPLETE": res = "success"
    elif r == "OFFSIDE": res = "offside"
    elif r in ["INCOMPLETE", "OUT"]: res = "fail"
    else: res = "fail"
    body = _parse_bodypart(b)
    return a, res, body

def _parse_shot_event(row):
    a = "shot"
    r = str(row.get("result", "") or "").upper()
    b = str(row.get("body_part_type", "") or "").upper()
    if r == "GOAL": res = "success"
    elif r == "OWN_GOAL": a = "bad_touch"; res = "owngoal"
    else: res = "fail"
    body = _parse_bodypart(b)
    return a, res, body

def _parse_take_on_event(_): return "take_on", "success", "foot"
def _parse_carry_event(_): return "dribble", "success", "foot"
def _parse_foul_event(row):
    card = str(row.get("result", "") or "").upper()
    if card == "YELLOW_CARD":
        return "foul", "yellow_card", "foot"
    elif card == "RED_CARD":
        return "foul", "red_card", "foot"
    return "foul", "fail", "foot"
def _parse_duel_event(row):
    result = str(row.get("result", "") or "").upper()
    res = "success" if result == "WON" else "fail"
    return "tackle", res, "foot"
def _parse_clearance_event(_): return "clearance", "success", "foot"
def _parse_miscontrol_event(_): return "bad_touch", "fail", "foot"
def _parse_goalkeeper_event(_): return "keeper_save", "success", "other"
def _parse_interception_event(_): return "interception", "success", "foot"
def _parse_bodypart(raw):
    raw = raw.upper() if isinstance(raw, str) else ""
    if "HEAD" in raw: return "head"
    elif "LEFT" in raw: return "foot_left"
    elif "RIGHT" in raw: return "foot_right"
    else: return "other"

In [3]:
EVENT_PARSERS = {
    "PASS": _parse_pass_event,
    "SHOT": _parse_shot_event,
    "TAKE_ON": _parse_take_on_event,
    "CARRY": _parse_carry_event,
    "FOUL_COMMITTED": _parse_foul_event,
    "DUEL": _parse_duel_event,
    "CLEARANCE": _parse_clearance_event,
    "MISCONTROL": _parse_miscontrol_event,
    "GOALKEEPER": _parse_goalkeeper_event,
    "INTERCEPTION": _parse_interception_event,
}

In [4]:
from datetime import datetime, timedelta

def timestamp_para_segundos(timestamp):
    # Separa a parte da fração de segundos
    timestamp_str = str(timestamp)
    partes = timestamp_str.split('.')
    tempo_sem_fracao = partes[0]
    
    # Converte a parte de horas, minutos e segundos para um objeto datetime
    # Usamos um ano, mês e dia arbitrários, pois só nos importam as horas, minutos e segundos
    # strptime converte uma string para um objeto datetime
    tempo_obj = datetime.strptime(tempo_sem_fracao, '%H:%M:%S')
    
    # Calcula os segundos totais a partir do objeto datetime
    total_segundos = tempo_obj.hour * 3600 + tempo_obj.minute * 60 + tempo_obj.second
    
    # Adiciona a fração de segundos (se houver)
    if len(partes) > 1:
        fracao_segundos_str = '0.' + partes[1]
        total_segundos += float(fracao_segundos_str)
        
    return total_segundos

### 3. Conversor Principal

In [ ]:
def convert_parquet_to_spadl(df):
    out = pd.DataFrame()
    out["game_id"] = df.get("game_id", df["match_id"])
    out["period_id"] = df["period_id"]
    out["team_id"] = df["team_id"]
    out["player_id"] = df["player_id"]
    out["start_x"] = df["coordinates_x"]
    out["start_y"] = df["coordinates_y"]
    out["end_x"] = df["end_coordinates_x"]
    out["end_y"] = df["end_coordinates_y"]
    out["original_event_id"] = df["event_id"]
    out["time_seconds"] = df["timestamp"].apply(timestamp_para_segundos)
    types, results, bodies = [], [], []
    for _, row in df.iterrows():
        event_type = str(row.get("event_type", "")).upper()
        parser = EVENT_PARSERS.get(event_type, lambda r: ("non_action", "success", "foot"))
        a, r, b = parser(row)
        types.append((spadlconfig.actiontypes.index(a), a))
        results.append((spadlconfig.results.index(r), r))
        bodies.append((spadlconfig.bodyparts.index(b), b))
    out[["type_id", "type_name"]] = pd.DataFrame(types, index=out.index)
    out[["result_id", "result_name"]] = pd.DataFrame(results, index=out.index)
    out[["bodypart_id", "bodypart_name"]] = pd.DataFrame(bodies, index=out.index)
    out = out[out.type_name != "non_action"].reset_index(drop=True)
    out = _fix_clearances(out)
    out["action_id"] = range(len(out))
    out = _add_dribbles(out)
    return out

In [9]:
parquet_path = "raw_events/eventos_com_generic.parquet"
df = pd.read_parquet(parquet_path)
spadl = convert_parquet_to_spadl(df)
spadl.to_parquet("spadl_field.parquet", index=False)
print("Arquivo salvo como 'spadl_field.parquet'")

Arquivo salvo como 'spadl_field.parquet'
